# GNN Pipeline — Market Basket Analysis
## Amazon Computers Dataset · Node Classification + Community Detection + Recommendation

> All model/training code lives in `src/gnn_market_basket/`.  
> This notebook **only imports and runs** — no repeated logic here.


## Step 0 — Point Python to the `src/` package

In [1]:
import os
import sys

# 1. This grabs the absolute path of the workspace folder open in VS Code
# Go up one level from 'notebooks' to get the main 'communityDetection' folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 2. Join the project root directly with your 'src' directory
src_path = os.path.join(project_root, "src")

# 3. Inject it at the very front of your system paths
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"✅ Absolute Project Root: {project_root}")
print(f"🚀 Python is now looking directly inside: {src_path}")

✅ Absolute Project Root: c:\Users\saisi\OneDrive\Desktop\communityDetection
🚀 Python is now looking directly inside: c:\Users\saisi\OneDrive\Desktop\communityDetection\src


## Step 1 — Import everything from `src/gnn_market_basket/`

In [2]:
# Step 1 import cell should be:
import torch
from gnn_dataset_implementation.dataset import load_data
from gnn_dataset_implementation.models import DeepGCN, StandardGAT, ResAttJKNet
from gnn_dataset_implementation.train_eval import train_model, evaluate_classification, evaluate_clustering
from gnn_dataset_implementation.recommend import recommend_products


# Now you can use them directly
print("All custom modular models loaded successfully into the workspace!")

All custom modular models loaded successfully into the workspace!


## Step 2 — Load Dataset

`load_data()` is defined in `src/gnn_market_basket/dataset.py`.  
It downloads Amazon Computers and creates 60/20/20 train/val/test masks.


In [3]:
data, num_classes = load_data()
in_channels = data.num_node_features

[Dataset] Nodes: 13752  |  Edges: 491722
[Dataset] Features: 767  |  Classes: 10
[Dataset] Train: 8251  Val: 2750  Test: 2751


## Step 3 — Baseline 1: DeepGCN

**Defined in:** `src/gnn_market_basket/models.py` → class `DeepGCN`

6 GCN layers with no residual connections.  
**Expected failure:** Over-smoothing — all embeddings collapse to the same vector.


In [4]:
HIDDEN = 64
EPOCHS = 150

deep_gcn = DeepGCN(in_channels, hidden_channels=HIDDEN,
                   out_channels=num_classes, num_layers=6)

deep_gcn = train_model(deep_gcn, data, epochs=EPOCHS, lr=0.005)

gcn_acc = evaluate_classification(deep_gcn, data)
gcn_ari = evaluate_clustering(deep_gcn, data, num_classes)

print(f"DeepGCN  →  Test Acc: {gcn_acc:.4f}  |  ARI: {gcn_ari:.4f}")

Training Model for 150 epochs ...
  Epoch  30 | Loss: 1.1532 | Val Acc: 0.6916
  Epoch  60 | Loss: 0.8866 | Val Acc: 0.7404
  Epoch  90 | Loss: 0.6814 | Val Acc: 0.7982
  Epoch 120 | Loss: 0.6115 | Val Acc: 0.8135
  Epoch 150 | Loss: 0.6179 | Val Acc: 0.8236
Model training complete ✓

DeepGCN  →  Test Acc: 0.8055  |  ARI: 0.2853


## Step 4 — Baseline 2: StandardGAT

**Defined in:** `src/gnn_market_basket/models.py` → class `StandardGAT`

2 GAT layers with 8 attention heads.  
**Expected failure:** Overfitting on this dense graph (~491k edges).


In [5]:
std_gat = StandardGAT(in_channels, hidden_channels=HIDDEN,
                      out_channels=num_classes, heads=8, dropout=0.6)

std_gat = train_model(std_gat, data, epochs=EPOCHS, lr=0.005)

gat_acc = evaluate_classification(std_gat, data)
gat_ari = evaluate_clustering(std_gat, data, num_classes)

print(f"StandardGAT  →  Test Acc: {gat_acc:.4f}  |  ARI: {gat_ari:.4f}")

Training Model for 150 epochs ...
  Epoch  30 | Loss: 1.3611 | Val Acc: 0.7647
  Epoch  60 | Loss: 0.8436 | Val Acc: 0.8447
  Epoch  90 | Loss: 0.6505 | Val Acc: 0.8764
  Epoch 120 | Loss: 0.5621 | Val Acc: 0.8920
  Epoch 150 | Loss: 0.5044 | Val Acc: 0.8978
Model training complete ✓

StandardGAT  →  Test Acc: 0.8884  |  ARI: 0.4932


## Step 5 — Novel Model: ResAttJKNet

**Defined in:** `src/gnn_market_basket/models.py` → class `ResAttJKNet`

Three innovations:
- **Dual-Perspective:** GCNConv + GATConv in parallel, averaged each layer
- **Residual:** `output = fused + input` — stops over-smoothing
- **Jumping Knowledge:** concatenate ALL layer outputs → multi-scale embedding


In [6]:
res_att_jk = ResAttJKNet(in_channels, hidden_channels=HIDDEN,
                          out_channels=num_classes,
                          num_layers=3, gat_heads=4, dropout=0.5)

res_att_jk = train_model(res_att_jk, data, epochs=EPOCHS, lr=0.005)

jk_acc = evaluate_classification(res_att_jk, data)
jk_ari = evaluate_clustering(res_att_jk, data, num_classes)

print(f"ResAttJKNet  →  Test Acc: {jk_acc:.4f}  |  ARI: {jk_ari:.4f}")

Training Model for 150 epochs ...
  Epoch  30 | Loss: 0.9050 | Val Acc: 0.7091
  Epoch  60 | Loss: 0.4971 | Val Acc: 0.8480
  Epoch  90 | Loss: 0.3348 | Val Acc: 0.8978
  Epoch 120 | Loss: 0.2484 | Val Acc: 0.9131
  Epoch 150 | Loss: 0.1992 | Val Acc: 0.9138
Model training complete ✓

ResAttJKNet  →  Test Acc: 0.9026  |  ARI: 0.4963


## Step 6 — Final Comparison Table

In [7]:
print("=" * 58)
print(f"{'Model':<22} {'Test Accuracy':>13} {'ARI':>10}")
print("-" * 58)
print(f"{'DeepGCN (6-layer)':<22} {gcn_acc:>13.4f} {gcn_ari:>10.4f}  ← over-smoothing")
print(f"{'StandardGAT':<22} {gat_acc:>13.4f} {gat_ari:>10.4f}  ← overfitting")
print(f"{'ResAttJKNet (ours)':<22} {jk_acc:>13.4f} {jk_ari:>10.4f}  ← best")
print("=" * 58)

Model                  Test Accuracy        ARI
----------------------------------------------------------
DeepGCN (6-layer)             0.8055     0.2853  ← over-smoothing
StandardGAT                   0.8884     0.4932  ← overfitting
ResAttJKNet (ours)            0.9026     0.4963  ← best


## Step 7 — Product Recommendation

**Defined in:** `src/gnn_market_basket/recommend.py` → `recommend_products()`

Uses cosine similarity on ResAttJKNet's multi-scale embeddings.  
Products with similar neighbourhood structure + attention patterns score highest.


In [8]:
TARGET = 42   # change this to any product node ID you want

recs = recommend_products(TARGET, res_att_jk, data, num_recommendations=5)

print(f"Target Product: #{TARGET}  (Category: {data.y[TARGET].item()})\n")
print(f"{'Rank':<6} {'Product ID':<12} {'Category':<12} {'Cosine Sim'}")
print("-" * 44)
for rank, (node_id, score) in enumerate(recs, 1):
    print(f"{rank:<6} #{node_id:<11} {data.y[node_id].item():<12} {score:.4f}")

Target Product: #42  (Category: 4)

Rank   Product ID   Category     Cosine Sim
--------------------------------------------
1      #9531        4            0.9885
2      #9303        4            0.9878
3      #7838        4            0.9872
4      #12660       4            0.9867
5      #7109        4            0.9866


## Module Map

| What you see here | Where the code actually lives |
|-------------------|-------------------------------|
| `load_data()` | `src/gnn_market_basket/dataset.py` |
| `DeepGCN` | `src/gnn_market_basket/models.py` |
| `StandardGAT` | `src/gnn_market_basket/models.py` |
| `ResAttJKNet` | `src/gnn_market_basket/models.py` |
| `train_model()` | `src/gnn_market_basket/train_eval.py` |
| `evaluate_classification()` | `src/gnn_market_basket/train_eval.py` |
| `evaluate_clustering()` | `src/gnn_market_basket/train_eval.py` |
| `recommend_products()` | `src/gnn_market_basket/recommend.py` |
